# ITELv5 - SECCIÓN 1: CONFIGURACIÓN Y ENTORNO

Prepara el entorno y rutas, fija semillas, valida dependencias y centraliza
utilidades para el pipeline de Deep Learning.

> **Nota de arquitectura:** Este notebook es la base de toda la cadena.
> Todos los demás lo invocan con `%run "./ITELv5_01_CONFIGURACION.ipynb"`.

In [1]:
# ===============================================================_
# 1.0 — Importaciones y utilidades básicas
# ===============================================================_
from pathlib import Path
import os, sys, json, random, importlib.util
import numpy as np
import torch

class ITELConfig:
    # ── Estructura del repo ITEL ──────────────────────────────────
    #
    #   ITEL/research/               ← BASE_DIR
    #   ├── Datos_Brutos/            ← DATA_DIR  (Healthy/ + Patients/)
    #   ├── processed_data/          ← PROCESADOS_DIR
    #   ├── reports/CSV/             ← CSV_DIR
    #   ├── reports/Figuras/         ← FIGURAS_DIR
    #   ├── logs/                    ← LOGS_DIR
    #   └── notebooks/               ← aquí vive este archivo
    #
    #   ITEL/assets/models/          ← MODELS_DIR
    #
    # ── Detección robusta de BASE_DIR ────────────────────────────
    # Funciona independientemente del CWD que configure el IDE:
    # busca la carpeta 'research' que contiene 'notebooks/' con este archivo.
    _anchor = 'ITELv5_01_CONFIGURACION.ipynb'   # archivo de referencia
    _cwd    = Path('.').resolve()

    if   (_cwd / _anchor).exists():                              # CWD = notebooks/
        BASE_DIR = _cwd.parent
    elif (_cwd / 'notebooks' / _anchor).exists():               # CWD = research/
        BASE_DIR = _cwd
    elif (_cwd / 'research' / 'notebooks' / _anchor).exists():  # CWD = ITEL/ root
        BASE_DIR = _cwd / 'research'
    else:                                                        # fallback
        BASE_DIR = _cwd.parent

    # Rutas derivadas
    DATA_DIR       = BASE_DIR / 'Datos_Brutos'
    PROCESADOS_DIR = BASE_DIR / 'processed_data'
    EMB_DIR        = PROCESADOS_DIR
    REPORTES_DIR   = BASE_DIR / 'reports'
    CSV_DIR        = REPORTES_DIR / 'CSV'
    FIGURAS_DIR    = REPORTES_DIR / 'Figuras'
    LOGS_DIR       = BASE_DIR / 'logs'
    MODELS_DIR     = BASE_DIR.parent / 'assets' / 'models'   # ITEL/assets/models/
    DATASET_DIR    = BASE_DIR
    SEED           = 42

    @staticmethod
    def set_seeds(seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

    @staticmethod
    def check_dirs():
        print('🔍 Verificando estructura de carpetas...')
        # Carpetas de salida: se crean si no existen
        for name, path in [
            ('processed_data',   ITELConfig.PROCESADOS_DIR),
            ('reports/CSV',      ITELConfig.CSV_DIR),
            ('reports/Figuras',  ITELConfig.FIGURAS_DIR),
            ('logs',             ITELConfig.LOGS_DIR),
            ('assets/models',    ITELConfig.MODELS_DIR),
        ]:
            if not path.exists():
                path.mkdir(parents=True, exist_ok=True)
                print(f'  📁 Creada: {name}')

        if not ITELConfig.DATA_DIR.exists():
            print(f'  ❌ No encontrada: {ITELConfig.DATA_DIR}')
            return False

        h = ITELConfig.DATA_DIR / 'Healthy'
        p = ITELConfig.DATA_DIR / 'Patients'
        if not h.exists() or not p.exists():
            print('  ⚠️  Faltan subcarpetas Healthy/ o Patients/ dentro de Datos_Brutos/')
            return False

        hc = sum(1 for _ in h.rglob('*.wav'))
        pc = sum(1 for _ in p.rglob('*.wav'))
        print(f'  ✅ Datos_Brutos: {hc} WAV Healthy  |  {pc} WAV Patients')
        return True

    @staticmethod
    def obtener_rutas_vocales():
        rutas = {'Healthy': [], 'Patients': []}
        subdirs = ['01SAMOHL','02SOUHL','03_1SL','04_2SL','05_3SL','06_4SL','07_VSL']
        for categoria in ['Healthy', 'Patients']:
            base = ITELConfig.DATA_DIR / categoria
            if not base.exists():
                continue
            for sujeto in base.iterdir():
                if not sujeto.is_dir():
                    continue
                for sub in subdirs:
                    p = sujeto / sub
                    if p.exists():
                        rutas[categoria].append(p)
        return rutas


def check_dependencies():
    required = ['librosa','torchaudio','sklearn','matplotlib',
                'seaborn','tqdm','soundfile','pandas','numpy','torch']
    missing = [p for p in required if not importlib.util.find_spec(p)]
    if missing:
        print(f'  ⚠️  Paquetes faltantes: {", ".join(missing)}')
        print('     Instala con: pip install -r research/requirements.txt')
    else:
        print('  ✅ Todas las dependencias disponibles.')


# ── Inicialización ────────────────────────────────────────────────
ITELConfig.set_seeds(ITELConfig.SEED)
ITELConfig.check_dirs()
check_dependencies()

print(f'\n📁 BASE_DIR : {ITELConfig.BASE_DIR}')
print(f'📂 DATA_DIR : {ITELConfig.DATA_DIR}')
print(f'🖥️  GPU      : {torch.cuda.is_available()}')


🔍 Verificando estructura de carpetas...
  ✅ Datos_Brutos: 1680 WAV Healthy  |  2173 WAV Patients
  ✅ Todas las dependencias disponibles.

📁 BASE_DIR : C:\Users\USER\Documents\GitHub\ITEL\research
📂 DATA_DIR : C:\Users\USER\Documents\GitHub\ITEL\research\Datos_Brutos
🖥️  GPU      : False


In [2]:
# ===============================================================_
# 1.1 — Resumen de rutas
# ===============================================================_
print('Rutas del proyecto ITELv5:')
print(f'  Datos_Brutos    : {ITELConfig.DATA_DIR}')
print(f'  Procesados      : {ITELConfig.PROCESADOS_DIR}')
print(f'  Modelos         : {ITELConfig.MODELS_DIR}')
print(f'  Reportes CSV    : {ITELConfig.CSV_DIR}')
print(f'  Reportes Figuras: {ITELConfig.FIGURAS_DIR}')
print(f'  Logs            : {ITELConfig.LOGS_DIR}')

rutas_vocales = ITELConfig.obtener_rutas_vocales()
print(f'\n🔍 Healthy  : {len(rutas_vocales.get("Healthy",  []))} subrutas de audio')
print(f'🔍 Patients : {len(rutas_vocales.get("Patients", []))} subrutas de audio')
print('\n✅ Configuración ITELv5 lista.')


Rutas del proyecto ITELv5:
  Datos_Brutos    : C:\Users\USER\Documents\GitHub\ITEL\research\Datos_Brutos
  Procesados      : C:\Users\USER\Documents\GitHub\ITEL\research\processed_data
  Modelos         : C:\Users\USER\Documents\GitHub\ITEL\assets\models
  Reportes CSV    : C:\Users\USER\Documents\GitHub\ITEL\research\reports\CSV
  Reportes Figuras: C:\Users\USER\Documents\GitHub\ITEL\research\reports\Figuras
  Logs            : C:\Users\USER\Documents\GitHub\ITEL\research\logs

🔍 Healthy  : 308 subrutas de audio
🔍 Patients : 378 subrutas de audio

✅ Configuración ITELv5 lista.
